# Capstone Project: Will It Rain Tomorrow in Australia?
**Dataset:** Rain in Australia (`weatherAUS.csv`), Kaggle
**Problem type:** Binary classification — predict `RainTomorrow` (Yes/No)


## Planning

1. **What am I predicting?** `RainTomorrow` -- binary classification (rain or no rain the next day).
2. **Columns available:** 22 features -- location, date, temperatures, rainfall, evaporation, sunshine,
   wind (speed + direction at multiple times), humidity, pressure, cloud cover, and whether it rained today.
   Expected strong predictors: `Humidity3pm`, `Pressure3pm`, `Sunshine`, `Cloud3pm`, `RainToday`.
3. **Size:** 145,460 rows x 23 columns.
4. **First-look cleaning needs:** heavy missingness in `Sunshine`/`Evaporation`/`Cloud9am`/`Cloud3pm`
   (35-48% missing), Yes/No columns need encoding, 16-direction wind columns need one-hot encoding,
   and ~2% of rows are missing the target itself.
5. **Useful version by tomorrow:** cleaned data, 3+ charts with insights, 2-3 trained & compared
   classifiers -- then (Day 18) a saved model behind a working Streamlit app with a clear README.

## Loading and First Exploration

A quick first look at the raw file: its shape, column types, and summary statistics.

In [2]:
import pandas as pd

df = pd.read_csv("weatherAUS.csv")
print(df.shape)
print(df.head())
print(df.info())
print(df.describe())

(145460, 23)
         Date Location  MinTemp  MaxTemp  Rainfall  Evaporation  Sunshine  \
0  2008-12-01   Albury     13.4     22.9       0.6          NaN       NaN   
1  2008-12-02   Albury      7.4     25.1       0.0          NaN       NaN   
2  2008-12-03   Albury     12.9     25.7       0.0          NaN       NaN   
3  2008-12-04   Albury      9.2     28.0       0.0          NaN       NaN   
4  2008-12-05   Albury     17.5     32.3       1.0          NaN       NaN   

  WindGustDir  WindGustSpeed WindDir9am  ... Humidity9am  Humidity3pm  \
0           W           44.0          W  ...        71.0         22.0   
1         WNW           44.0        NNW  ...        44.0         25.0   
2         WSW           46.0          W  ...        38.0         30.0   
3          NE           24.0         SE  ...        45.0         16.0   
4           W           41.0        ENE  ...        82.0         33.0   

   Pressure9am  Pressure3pm  Cloud9am  Cloud3pm  Temp9am  Temp3pm  RainToday  \
0    

## Cleaning the Dataset

Drop rows with no target, pull the month out of the date, encode the Yes/No columns as 0/1, fill missing numeric values with the column median, and fill missing wind directions with the most common value.

In [3]:
import numpy as np

print("Missing values before cleaning:")
print(df.isnull().sum().sort_values(ascending=False))
print("\nDuplicate rows:", df.duplicated().sum())

# Drop rows with an unknown 
df = df.dropna(subset=["RainTomorrow", "RainToday"]).copy()

# Month
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.month
df = df.drop(columns=["Date"])

# Simple mapping
df["RainToday"] = df["RainToday"].map({"No": 0, "Yes": 1})
df["RainTomorrow"] = df["RainTomorrow"].map({"No": 0, "Yes": 1})

# Median-fill remaining numeric missing values
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if c not in ["RainToday", "RainTomorrow", "Month"]]
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Mode-fill missing wind directions
for col in ["WindGustDir", "WindDir9am", "WindDir3pm"]:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values remaining:", df.isnull().sum().sum())
print("Final shape:", df.shape)

Missing values before cleaning:
Sunshine         69835
Evaporation      62790
Cloud3pm         59358
Cloud9am         55888
Pressure9am      15065
Pressure3pm      15028
WindDir9am       10566
WindGustDir      10326
WindGustSpeed    10263
Humidity3pm       4507
WindDir3pm        4228
Temp3pm           3609
RainTomorrow      3267
Rainfall          3261
RainToday         3261
WindSpeed3pm      3062
Humidity9am       2654
WindSpeed9am      1767
Temp9am           1767
MinTemp           1485
MaxTemp           1261
Date                 0
Location             0
dtype: int64

Duplicate rows: 0
Missing values remaining: 0
Final shape: (140787, 23)


## Engineering 9am -> 3pm Change Features

The raw 9am and 3pm columns are two separate snapshots, but the *change* between them is often more predictive than either reading alone (e.g. a falling pressure through the day is a classic sign that rain is approaching). New features are derived here: `PressureChange`, `HumidityChange`, `TempChange`, `CloudChange`, and `WindSpeedChange`, each computed as (3pm value - 9am value). The original 9am/3pm columns are kept too, so the model (and the correlation heatmap below) can show whether the raw readings or the derived trends carry more signal.

In [10]:
import pandas as pd

df = pd.read_csv("weatherAUS_cleaned.csv")

# 1. Atmospheric Instability Features (Deltas)
df["Pressure_Trend"] = df["Pressure3pm"] - df["Pressure9am"]
df["Temp_Change"] = df["Temp3pm"] - df["Temp9am"]
df["Humidity_Drop"] = df["Humidity9am"] - df["Humidity3pm"]

# 2. Daily Averages
df["Pressure_Mean"] = (df["Pressure9am"] + df["Pressure3pm"]) / 2
df["Cloud_Mean"] = (df["Cloud9am"] + df["Cloud3pm"]) / 2
df["WindSpeed_Mean"] = (df["WindSpeed9am"] + df["WindSpeed3pm"]) / 2

# Drop redundant raw 9am/3pm columns to keep dataset light
cols_to_drop = [
    "Temp9am",
    "Temp3pm",
    "Pressure9am",
    "Pressure3pm",
    "Cloud9am",
    "Cloud3pm",
    "WindSpeed9am",
    "WindSpeed3pm",
    "Humidity9am",
]
df_reduced = df.drop(columns=cols_to_drop)

In [11]:
import pandas as pd

# 1. Load the cleaned dataset
df = pd.read_csv("weatherAUS_cleaned.csv")

# 2. Create the reduced/merged features
df["Temp_Change"] = df["Temp3pm"] - df["Temp9am"]
df["Pressure_Trend"] = df["Pressure3pm"] - df["Pressure9am"]
df["Humidity_Drop"] = df["Humidity9am"] - df["Humidity3pm"]

df["Pressure_Mean"] = (df["Pressure9am"] + df["Pressure3pm"]) / 2
df["Cloud_Mean"] = (df["Cloud9am"] + df["Cloud3pm"]) / 2
df["WindSpeed_Mean"] = (df["WindSpeed9am"] + df["WindSpeed3pm"]) / 2

# 3. View side-by-side comparison of original vs engineered features
preview_cols = [
    "Location",
    "Temp9am",
    "Temp3pm",
    "Temp_Change",
    "Pressure9am",
    "Pressure3pm",
    "Pressure_Trend",
    "Humidity9am",
    "Humidity3pm",
    "Humidity_Drop",
    "RainTomorrow",
]

print("=== RAW vs ENGINEERED FEATURES (Sample Rows) ===")
print(df[preview_cols].head(5).to_string(index=False))

# 4. Inspect overall statistical changes across the dataset
print("\n=== SUMMARY STATISTICS OF NEW FEATURES ===")
new_features = [
    "Temp_Change",
    "Pressure_Trend",
    "Humidity_Drop",
    "Pressure_Mean",
    "Cloud_Mean",
    "WindSpeed_Mean",
]
print(
    df[new_features]
    .describe()
    .T[["mean", "std", "min", "50%", "max"]]
    .round(2)
)

=== RAW vs ENGINEERED FEATURES (Sample Rows) ===
Location  Temp9am  Temp3pm  Temp_Change  Pressure9am  Pressure3pm  Pressure_Trend  Humidity9am  Humidity3pm  Humidity_Drop  RainTomorrow
  Albury     16.9     21.8          4.9       1007.7       1007.1            -0.6           71           22             49             0
  Albury     17.2     24.3          7.1       1010.6       1007.8            -2.8           44           25             19             0
  Albury     21.0     23.2          2.2       1007.6       1008.7             1.1           38           30              8             0
  Albury     18.1     26.5          8.4       1017.6       1012.8            -4.8           45           16             29             0
  Albury     17.8     29.7         11.9       1010.8       1006.0            -4.8           82           33             49             0

=== SUMMARY STATISTICS OF NEW FEATURES ===
                   mean    std     min     50%      max
Temp_Change        4.70   3.7

In [13]:
import pandas as pd

# 1. Load cleaned dataset
df = pd.read_csv("weatherAUS_cleaned.csv")

# 2. Engineer merged/reduced features
df["Pressure_Trend"] = df["Pressure3pm"] - df["Pressure9am"]
df["Temp_Change"] = df["Temp3pm"] - df["Temp9am"]
df["Humidity_Drop"] = df["Humidity9am"] - df["Humidity3pm"]

df["Pressure_Mean"] = (df["Pressure9am"] + df["Pressure3pm"]) / 2
df["Cloud_Mean"] = (df["Cloud9am"] + df["Cloud3pm"]) / 2
df["WindSpeed_Mean"] = (df["WindSpeed9am"] + df["WindSpeed3pm"]) / 2

# 3. Drop redundant 9 AM and 3 PM raw feature columns
cols_to_drop = [
    "Temp9am",
    "Temp3pm",
    "Pressure9am",
    "Pressure3pm",
    "Cloud9am",
    "Cloud3pm",
    "WindSpeed9am",
    "WindSpeed3pm",
    "Humidity9am",
    "Humidity3pm",
]
df_reduced = df.drop(columns=cols_to_drop)

# 4. Save to CSV file
df_reduced.to_csv("weatherAUS_reduced.csv", index=False)

In [ ]:
df = pd. read_cv("weatherAUS_cleaned.csv")
df["Pressure_Trend"] = df["temp3pm"] -df["temp9am"]
df["humidity_drop"] = df["Humidity9am"]
df["Humidity_Drop"] = df["Humidity9am"]+df["Pressure3pm"]
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardaScaler



In [15]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

# Load the engineered dataframe from previous steps
df = pd.read_csv("weatherAUS_cleaned.csv") 

# One-hot encode categorical features (Location, Wind directions)
categorical_cols = ["Location", "WindGustDir", "WindDir9am", "WindDir3pm"]
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separate features (X) and target (y)
X = df_encoded.drop(columns=["RainTomorrow"])
y = df_encoded["RainTomorrow"]

# Train/Test Split (Stratified on target due to ~22% positive class imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save feature column names for Streamlit app inference
model_columns = list(X.columns)
joblib.dump(model_columns, "model_columns.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [ ]:
ratio = (len(y_train) - sum(y_train)) / sum(y_train)

models = {
    "logistic regression": LogisticRegression(max_iter = 1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    "XGBoost":XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.1, scale_pos_weight=ration, random_state=42, n_jobs=-1)
}
results = []
for name, model in models.items():
    X_tr = X_train_scaled if name == "Logistic Regression" else X_train
    X_te = X_test_scaled if name == "Logistic Regression" else X_test

    model.fit(X_tr, y_train)

    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    acc = accuracy_score(y_test, y_train)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    results.append({
    "Model": name,
    "Accuracy": round(acc, 4),
    "Precision": round(prec, 4),
    "Recall": round(rec, 4),
    "F1-Score": round(roc_auc, 4)
    })

    print(f"=== {name} Classification 